<a href="https://colab.research.google.com/github/jm651120/UK-Housing-Green-Premium/blob/main/01_Data_Integration_and_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Data Integration and Preprocessing
**Mestrado em Data Science | Nova IMS**
**Tópico:** Decoding the UK Housing Market (2021-2025)

---

## Passo 1: Configuração do Ambiente e Acesso aos Dados
Nesta etapa, estabelecemos a ligação entre o Google Colab e o Google Drive onde residem os dados *Open Government Data* (OGD). O objetivo é garantir que o ambiente tem acesso de leitura aos ficheiros originais do *Land Registry* (LR-PPD), *Energy Performance Certificates* (EPC) e *Office for National Statistics* (ONS).

In [ ]:
# Importar a biblioteca para ligar ao Google Drive
from google.colab import drive
import os

# 1. Montar o Google Drive (vai pedir-te para fazeres login com o teu Google e autorizar)
drive.mount('/content/drive')

# 2. Definir o caminho para a tua pasta
# Nota: O Colab traduz "O meu disco" para "MyDrive"
caminho_pasta = '/content/drive/MyDrive/Tese/UK DataSets/'

# 3. Listar os ficheiros para garantir que o Colab está a ver a pasta certa
print("\n--- Ficheiros encontrados na pasta ---")
try:
    ficheiros = os.listdir(caminho_pasta)
    for ficheiro in ficheiros:
        print(f"✅ {ficheiro}")
except FileNotFoundError:
    print("Erro: A pasta não foi encontrada.")

Mounted at /content/drive

--- Ficheiros encontrados na tua pasta ---
✅ ONSPD_NOV_2025_UK.csv
✅ certificates.csv
✅ columns.csv
✅ pp-2025.csv
✅ pp-2024.csv
✅ pp-2023.csv
✅ pp-2022.csv
✅ pp-2021.csv


## Passo 2: Carregamento e Filtragem do LR-PPD (2021-2025)
Os ficheiros raw do *HM Land Registry Price Paid Data* não contêm cabeçalho (header). Iremos atribuir os nomes oficiais das colunas e proceder a um filtro geográfico imediato.
Para viabilizar a computação e alinhar os dados com o *Research Design* (Secção 3.2.1), os milhões de transações nacionais serão reduzidos apenas a três regiões:
* **GREATER LONDON** (Hyper-dense stress test)
* **GREATER MANCHESTER** (Urban baseline)
* **CORNWALL** (Rural/Coastal environment)

In [ ]:
import pandas as pd

# 1. Definir os nomes oficiais das colunas do HM Land Registry
colunas_lr = [
    'Transaction_ID', 'Price', 'Date_of_Transfer', 'Postcode', 'Property_Type',
    'Old_New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town_City',
    'District', 'County', 'PPD_Category', 'Record_Status'
]

# 2. Lista dos ficheiros anuais que tens no Drive
ficheiros_ppd = ['pp-2021.csv', 'pp-2022.csv', 'pp-2023.csv', 'pp-2024.csv', 'pp-2025.csv']

# 3. Zonas de estudo exatas
zonas_estudo = ['GREATER LONDON', 'GREATER MANCHESTER', 'CORNWALL']

# Lista vazia para ir guardando os dados de cada ano
dfs_filtrados = []

print("A iniciar o carregamento e filtragem espacial do LR-PPD.\n")

# 4. Loop para ler e filtrar cada ficheiro ano a ano
for ficheiro in ficheiros_ppd:
    caminho_ficheiro = caminho_pasta + ficheiro
    print(f"A processar o ficheiro: {ficheiro}...")

    # Ler o CSV (avisamos o pandas que não há cabeçalho e damos a nossa lista de nomes)
    df_temp = pd.read_csv(caminho_ficheiro, header=None, names=colunas_lr)

    # Manter apenas as linhas em que a coluna 'County' pertence às nossas zonas de estudo
    # Usamos str.upper() e str.strip() para garantir que não há erros de espaços ou minúsculas
    df_temp_filtrado = df_temp[df_temp['County'].astype(str).str.upper().str.strip().isin(zonas_estudo)]

    dfs_filtrados.append(df_temp_filtrado)

# 5. Juntar todos os anos num único dataset
df_transacoes = pd.concat(dfs_filtrados, ignore_index=True)

print("\n✅ Concluído com sucesso!")
print(f"Tamanho do Dataset (Número total de transações isoladas nas 3 zonas): {len(df_transacoes)} linhas.")

# Mostrar as primeiras 3 linhas para confirmarmos o aspeto dos dados
display(df_transacoes.head(3))

A iniciar o carregamento e filtragem espacial do LR-PPD. Isto pode demorar 1 ou 2 minutos...

A processar o ficheiro: pp-2021.csv...
A processar o ficheiro: pp-2022.csv...
A processar o ficheiro: pp-2023.csv...
A processar o ficheiro: pp-2024.csv...
A processar o ficheiro: pp-2025.csv...

✅ Concluído com sucesso!
Tamanho do Dataset (Número total de transações isoladas nas 3 zonas): 814067 linhas.


,Transaction_ID,Price,Date_of_Transfer,Postcode,Property_Type,Old_New,Duration,PAON,SAON,Street,Locality,Town_City,District,County,PPD_Category,Record_Status
0,{D707E535-3C78-0AD9-E053-6B04A8C067CC},226500,2021-08-12 00:00,WA15 8RF,F,N,L,16,NaN,REGENCY COURT,HALE,ALTRINCHAM,TRAFFORD,GREATER MANCHESTER,A,A
1,{D707E535-3C79-0AD9-E053-6B04A8C067CC},88820,2021-09-17 00:00,OL4 2EA,T,N,L,115,NaN,REDGRAVE STREET,NaN,OLDHAM,OLDHAM,GREATER MANCHESTER,A,A
2,{D707E535-3C7A-0AD9-E053-6B04A8C067CC},225000,2021-08-18 00:00,BL7 9DY,F,N,L,18,NaN,VALLEY MILL,NaN,BOLTON,BOLTON,GREATER MANCHESTER,A,A


## Passo 3: Carregamento Massivo dos Certificados Energéticos (EPC)
Para garantir a representatividade espacial exigida (Greater London, Greater Manchester e Cornwall), o dataset EPC é construído através da agregação de múltiplos ficheiros regionais. Para otimizar a memória RAM (prevenindo *Out of Memory errors*), aplicamos a técnica de *Load & Filter*, lendo exclusivamente as variáveis estruturais e de identificação definidas na secção 3.2.4 da metodologia.

In [ ]:
import pandas as pd
import os
import glob

print("A iniciar o carregamento massivo de Certificados Energéticos (EPC)...")

# 1. Caminho para a nova pasta que criaste com as 44 subpastas
# ATENÇÃO: Confirma se o nome da pasta no Drive é exatamente 'EPC_Folders'
caminho_epc_folders = caminho_pasta + 'EPC_Folders/'

# 2. As únicas colunas que precisamos de ler (Poupança extrema de RAM!)
colunas_necessarias_epc = [
    'POSTCODE', 'ADDRESS1', 'TOTAL_FLOOR_AREA',
    'NUMBER_HABITABLE_ROOMS', 'CURRENT_ENERGY_RATING', 'CONSTRUCTION_AGE_BAND'
]

# 3. Encontrar todos os ficheiros certificates.csv dentro das subpastas
padrao_busca = os.path.join(caminho_epc_folders, '**', 'certificates.csv')
ficheiros_epc = glob.glob(padrao_busca, recursive=True)

print(f"Encontrados {len(ficheiros_epc)} ficheiros de certificados. A processar...\n")

dfs_epc = []

# 4. Loop para ler cada ficheiro, filtrar e guardar
for ficheiro in ficheiros_epc:
    # Extrair o nome da pasta (ex: domestic-E09000033-Westminster) para vermos o progresso
    nome_pasta = os.path.basename(os.path.dirname(ficheiro))
    print(f"A ler: {nome_pasta}...")

    try:
        # Ler apenas as 6 colunas vitais
        df_temp = pd.read_csv(ficheiro, usecols=colunas_necessarias_epc, low_memory=False)
        dfs_epc.append(df_temp)
    except Exception as e:
        print(f"⚠️ Erro ao ler {nome_pasta}: {e}")

# 5. Juntar os 44 dataframes num único Super Dataset
print("\nA fundir todos os ficheiros regionais num só...")
df_epc = pd.concat(dfs_epc, ignore_index=True)

print("\n✅ Carregamento do Super Dataset EPC concluído com sucesso!")
print(f"Tamanho Total do Dataset EPC: {len(df_epc)} linhas.")

# 6. Mostrar as primeiras 3 linhas
display(df_epc.head(3))

A iniciar o carregamento massivo de Certificados Energéticos (EPC)...
Encontrados 44 ficheiros de certificados. A processar...

A ler: domestic-E06000052-Cornwall...
A ler: domestic-E08000002-Bury...
A ler: domestic-E08000007-Stockport...
A ler: domestic-E08000009-Trafford...
A ler: domestic-E08000005-Rochdale...
A ler: domestic-E08000008-Tameside...
A ler: domestic-E08000001-Bolton...
A ler: domestic-E08000003-Manchester...
A ler: domestic-E08000004-Oldham...
A ler: domestic-E08000010-Wigan...
A ler: domestic-E09000002-Barking-and-Dagenham...
A ler: domestic-E09000033-Westminster...
A ler: domestic-E09000009-Ealing...
A ler: domestic-E09000008-Croydon...
A ler: domestic-E09000011-Greenwich...
A ler: domestic-E09000006-Bromley...
A ler: domestic-E09000032-Wandsworth...
A ler: domestic-E09000005-Brent...
A ler: domestic-E09000007-Camden...
A ler: domestic-E09000010-Enfield...
A ler: domestic-E09000004-Bexley...
A ler: domestic-E09000027-Richmond-upon-Thames...
A ler: domestic-E09000026-

,ADDRESS1,POSTCODE,CURRENT_ENERGY_RATING,TOTAL_FLOOR_AREA,NUMBER_HABITABLE_ROOMS,CONSTRUCTION_AGE_BAND
0,4 Tregurtha Farm,TR17 0DR,D,68.58,4.0,England and Wales: 1983-1990
1,4 East Park,PL15 7EW,C,80.20,NaN,NO DATA!
2,Beatrice Cottage,PL14 4QX,D,81.37,3.0,England and Wales: 1967-1975


## Passo 4: Data Linkage (Integração LR-PPD e EPC)
Nesta fase, materializamos a integração descrita na secção 3.2.4 da tese. Para evitar duplicação cruzada (cross-join) num mesmo código postal, criamos uma chave de ligação determinística (`Merge_Key`) combinando o Código Postal limpo e o identificador primário da morada (o número ou nome da porta).
* **LR-PPD:** Utilizamos a coluna `PAON`.
* **EPC:** Extraímos o primeiro elemento da coluna `ADDRESS1` (antes da primeira vírgula ou espaço).
Após o *Inner Join*, removemos duplicados para garantir que a unidade de análise (a transação individual) se mantém intacta.

In [ ]:
import pandas as pd
import re

print(f"A iniciar a fusão de {len(df_transacoes)} transações com {len(df_epc)} certificados...")

# 1. Limpeza de Postcodes (Garantir que WA15 8RF se torna WA158RF)
df_transacoes['Postcode_Clean'] = df_transacoes['Postcode'].astype(str).str.replace(' ', '').str.upper()
df_epc['Postcode_Clean'] = df_epc['POSTCODE'].astype(str).str.replace(' ', '').str.upper()

# 2. Extração do Número da Porta (Regex para capturar o primeiro número da morada)
df_transacoes['Door_Number'] = df_transacoes['PAON'].astype(str).str.extract(r'(\d+)')
df_epc['Door_Number'] = df_epc['ADDRESS1'].astype(str).str.extract(r'(\d+)')

# 3. Criação da Chave de Ligação Única (Ex: M408NS_37)
df_transacoes['Merge_Key'] = df_transacoes['Postcode_Clean'] + "_" + df_transacoes['Door_Number'].fillna('UNKNOWN')
df_epc['Merge_Key'] = df_epc['Postcode_Clean'] + "_" + df_epc['Door_Number'].fillna('UNKNOWN')

# 4. Execução do Merge (Inner Join)
# Juntamos pelas duas chaves para garantir que o Postcode_Clean não duplica colunas (_x, _y)
print("A fundir os datasets.")
df_merged = pd.merge(df_transacoes, df_epc, on=['Merge_Key', 'Postcode_Clean'], how='inner')

# 5. Limpeza de Duplicados Rigorosa
# Mantemos a transação mais recente se a mesma chave aparecer várias vezes
df_merged = df_merged.sort_values('Date_of_Transfer').drop_duplicates(subset=['Transaction_ID'], keep='last')

print("\n✅ Fusão concluída com sucesso!")
print(f"Tamanho do Dataset Integrado: {len(df_merged)} linhas.")

# 6. Seleção das colunas finais definidas na Metodologia (Secção 3.2.4)
colunas_finais = [
    'Transaction_ID', 'Price', 'Date_of_Transfer', 'Postcode_Clean', 'Merge_Key',
    'County', 'Property_Type', 'Old_New', 'Duration', 'TOTAL_FLOOR_AREA',
    'NUMBER_HABITABLE_ROOMS', 'CURRENT_ENERGY_RATING', 'CONSTRUCTION_AGE_BAND'
]

df_final = df_merged[colunas_finais].copy()

# Renomear para o padrão da tese (British English)
df_final = df_final.rename(columns={'Duration': 'Tenure_Duration'})

display(df_final.head(5))

A iniciar a fusão de 814067 transações com 6141167 certificados...
A fundir os datasets. Com 6 milhões de linhas, isto pode demorar 1 ou 2 minutos...

✅ Fusão concluída com sucesso!
Tamanho do Dataset Integrado: 622649 linhas.


,Transaction_ID,Price,Date_of_Transfer,Postcode_Clean,Merge_Key,County,Property_Type,Old_New,Tenure_Duration,TOTAL_FLOOR_AREA,NUMBER_HABITABLE_ROOMS,CURRENT_ENERGY_RATING,CONSTRUCTION_AGE_BAND
351952,{BEF7EBBF-3E58-7A76-E053-6B04A8C092F7},590000,2021-01-01 00:00,BR60EN,BR60EN_47,GREATER LONDON,S,N,F,104.18,5.0,E,England and Wales: 1900-1929
347189,{BC8936BC-4A47-0E2C-E053-6C04A8C0DBF4},116000,2021-01-01 00:00,OL26JL,OL26JL_10,GREATER MANCHESTER,O,N,F,73.00,4.0,D,England and Wales: 1983-1990
8598,{E073986B-B017-2134-E053-6C04A8C0233B},290000,2021-01-01 00:00,CR02XT,CR02XT_145,GREATER LONDON,F,N,L,67.00,2.0,C,England and Wales: 2012 onwards
157858,{D707E534-B99D-0AD9-E053-6B04A8C067CC},669999,2021-01-01 00:00,E28GW,E28GW_8,GREATER LONDON,F,Y,L,54.00,NaN,B,NaN
339989,{BEF7EBBF-40CD-7A76-E053-6B04A8C092F7},360000,2021-01-01 00:00,BR49PU,BR49PU_2,GREATER LONDON,F,N,L,59.00,NaN,B,NO DATA!


## Passo 5: Integração Geoespacial (ONS)
Para permitir que os modelos de *Machine Learning* capturem as variações de preço hiperlocais (em contraste com as fronteiras artificiais dos Postcode Outcodes dos modelos Hedónicos), procedemos à integração das coordenadas geográficas exatas.
Utilizamos o *Office for National Statistics Postcode Directory* (ONSPD). Devido à dimensão do dataset nacional, a extração é otimizada na memória carregando exclusivamente as colunas de Código Postal, Latitude e Longitude, fundindo-as de seguida com o nosso dataset analítico através de um *Left Join*.

In [ ]:
import pandas as pd

print("A carregar os dados geográficos do ONS (Modo Otimizado para poupar RAM)...")

caminho_ons = caminho_pasta + 'ONSPD_NOV_2025_UK.csv'

# 1. Ler APENAS as colunas que interessam do ficheiro gigante de 1.35 GB
# O ONSPD normalmente usa 'pcds' para código postal, e 'lat'/'long' para coordenadas
# Usamos usecols para não crashar o Colab
try:
    df_ons = pd.read_csv(caminho_ons, usecols=['pcds', 'lat', 'long'])
except ValueError:
    # Se os nomes das colunas no teu ficheiro forem ligeiramente diferentes, o Pandas avisa
    print("Os nomes das colunas no ficheiro ONS podem ser diferentes. A ler a primeira linha para confirmar...")
    df_ons_amostra = pd.read_csv(caminho_ons, nrows=1)
    print(df_ons_amostra.columns.tolist())
    raise SystemExit("Pára o código aqui e envia a lista de colunas impressa ao teu assistente!")

# 2. Limpar o Postcode do ONS para ficar exatamente igual ao nosso Postcode_Clean
df_ons['Postcode_Clean'] = df_ons['pcds'].astype(str).str.replace(' ', '').str.upper()

# 3. Remover duplicados de código postal no ONS (por segurança)
df_ons = df_ons.drop_duplicates(subset=['Postcode_Clean'])

print("A fundir as coordenadas geográficas com o nosso dataset principal (Left Join)...")

# 4. Fazer o Merge (Left Join - porque não queremos perder casas, só queremos adicionar a localização)
df_final_geo = pd.merge(df_final, df_ons[['Postcode_Clean', 'lat', 'long']], on='Postcode_Clean', how='left')

# 5. Renomear para ficar elegante e em inglês
df_final_geo = df_final_geo.rename(columns={'lat': 'Latitude', 'long': 'Longitude'})

print("\n✅ Integração Geoespacial concluída!")
print(f"Tamanho do Dataset Final: {len(df_final_geo)} linhas e {len(df_final_geo.columns)} colunas.")

# Mostrar as primeiras linhas para vermos a magia da Latitude e Longitude adicionadas!
display(df_final_geo.head(5))

A carregar os dados geográficos do ONS (Modo Otimizado para poupar RAM)...
A fundir as coordenadas geográficas com o nosso dataset principal (Left Join)...

✅ Integração Geoespacial concluída!
Tamanho do Dataset Final: 622649 linhas e 15 colunas.


,Transaction_ID,Price,Date_of_Transfer,Postcode_Clean,Merge_Key,County,Property_Type,Old_New,Tenure_Duration,TOTAL_FLOOR_AREA,NUMBER_HABITABLE_ROOMS,CURRENT_ENERGY_RATING,CONSTRUCTION_AGE_BAND,Latitude,Longitude
0,{BEF7EBBF-3E58-7A76-E053-6B04A8C092F7},590000,2021-01-01 00:00,BR60EN,BR60EN_47,GREATER LONDON,S,N,F,104.18,5.0,E,England and Wales: 1900-1929,51.379582,0.099117
1,{BC8936BC-4A47-0E2C-E053-6C04A8C0DBF4},116000,2021-01-01 00:00,OL26JL,OL26JL_10,GREATER MANCHESTER,O,N,F,73.00,4.0,D,England and Wales: 1983-1990,53.563072,-2.111319
2,{E073986B-B017-2134-E053-6C04A8C0233B},290000,2021-01-01 00:00,CR02XT,CR02XT_145,GREATER LONDON,F,N,L,67.00,2.0,C,England and Wales: 2012 onwards,51.386351,-0.098812
3,{D707E534-B99D-0AD9-E053-6B04A8C067CC},669999,2021-01-01 00:00,E28GW,E28GW_8,GREATER LONDON,F,Y,L,54.00,NaN,B,NaN,51.531241,-0.071226
4,{BEF7EBBF-40CD-7A76-E053-6B04A8C092F7},360000,2021-01-01 00:00,BR49PU,BR49PU_2,GREATER LONDON,F,N,L,59.00,NaN,B,NO DATA!,51.376376,-0.021055


## Passo 6: Data Preprocessing (Missing Values & Outliers)
De acordo com as boas práticas do CRISP-ML, esta fase trata as anomalias dos dados em três frentes:
1. **Missing Values (Valores em falta):** Variáveis críticas (`CURRENT_ENERGY_RATING`, `TOTAL_FLOOR_AREA`, `Latitude`) com valores nulos são removidas. Para `NUMBER_HABITABLE_ROOMS`, aplica-se a imputação pela mediana. Categorias como "NO DATA!" em `CONSTRUCTION_AGE_BAND` são reclassificadas como "Unknown".
2. **Outlier Removal (Remoção de Extremos):** Aplica-se o método *Interquartile Range* (IQR) às variáveis contínuas de `Price` e `TOTAL_FLOOR_AREA` para eliminar anomalias estatísticas que distorcem algoritmos não-lineares.
3. **Summary Statistics:** Geração da tabela de estatísticas descritivas para validar a integridade do dataset final antes do *Encoding* e *Modelling*.

In [ ]:
import pandas as pd
import numpy as np

print("--- 1. Análise Inicial de Valores em Falta (NaNs) ---")
print(df_final_geo.isnull().sum())

print("\n--- 2. A Iniciar a Limpeza e Imputação ---")
df_clean = df_final_geo.copy()

# A. Tratar a Idade de Construção
df_clean['CONSTRUCTION_AGE_BAND'] = df_clean['CONSTRUCTION_AGE_BAND'].replace('NO DATA!', 'Unknown').fillna('Unknown')

# B. Imputar Quartos (Habitable Rooms) com a Mediana
mediana_quartos = df_clean['NUMBER_HABITABLE_ROOMS'].median()
df_clean['NUMBER_HABITABLE_ROOMS'] = df_clean['NUMBER_HABITABLE_ROOMS'].fillna(mediana_quartos)

# C. Aplicar Regras de Negócio do Mercado Imobiliário (Business Rules)
# 1. Remover preços abaixo de £50.000 (vendas não-mercado)
df_clean = df_clean[df_clean['Price'] >= 50000]

# 2. Quartos: Limpar erros a zero e limites máximos absurdos (manter entre 1 e 10 divisões)
df_clean = df_clean[(df_clean['NUMBER_HABITABLE_ROOMS'] >= 1) & (df_clean['NUMBER_HABITABLE_ROOMS'] <= 10)]

# 3. Área: Limpar casas com 0 m2 (manter apenas áreas realistas >= 15 m2)
df_clean = df_clean[df_clean['TOTAL_FLOOR_AREA'] >= 15]

# D. Remover linhas onde faltam Coordenadas, Rating de Energia ou Área
df_clean = df_clean.dropna(subset=['CURRENT_ENERGY_RATING', 'TOTAL_FLOOR_AREA', 'Latitude', 'Longitude'])

print("--- 3. Remoção de Outliers Superiores (Método IQR) ---")
def remove_top_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    return df[df[column] <= upper_bound]

# Aplicar o filtro ao Preço e ao Tamanho da casa
df_clean = remove_top_outliers_iqr(df_clean, 'Price')
df_clean = remove_top_outliers_iqr(df_clean, 'TOTAL_FLOOR_AREA')

print(f"✅ Limpeza Concluída! Sobraram {len(df_clean)} casas no mercado 'normal'.")

print("\n--- 4. Summary Statistics Table (Atualizada) ---")
tabela_estatisticas = df_clean[['Price', 'TOTAL_FLOOR_AREA', 'NUMBER_HABITABLE_ROOMS', 'Latitude', 'Longitude']].describe().round(2)
display(tabela_estatisticas)

--- 1. Análise Inicial de Valores em Falta (NaNs) ---
Transaction_ID                0
Price                         0
Date_of_Transfer              0
Postcode_Clean                0
Merge_Key                     0
County                        0
Property_Type                 0
Old_New                       0
Tenure_Duration               0
TOTAL_FLOOR_AREA              0
NUMBER_HABITABLE_ROOMS    70739
CURRENT_ENERGY_RATING         0
CONSTRUCTION_AGE_BAND      6398
Latitude                      3
Longitude                     3
dtype: int64

--- 2. A Iniciar a Limpeza e Imputação ---
--- 3. Remoção de Outliers Superiores (Método IQR) ---
✅ Limpeza Concluída! Sobraram 547948 casas no mercado 'normal'.

--- 4. Summary Statistics Table (Atualizada) ---


,Price,TOTAL_FLOOR_AREA,NUMBER_HABITABLE_ROOMS,Latitude,Longitude
count,547948.00,547948.00,547948.00,547948.00,547948.00
mean,411927.76,82.89,4.22,52.06,-1.18
std,226344.05,26.53,1.25,1.05,1.49
min,50000.00,15.00,1.00,49.96,-5.70
25%,233950.00,64.00,3.00,51.45,-2.25
50%,377000.00,80.00,4.00,51.56,-0.28
75%,545000.00,99.00,5.00,53.44,-0.07
max,1162500.00,158.22,10.00,53.68,0.31


### 7. Feature Engineering (Alinhamento com Secção 3.2.4)

De acordo com a metodologia, extraímos aqui as **variáveis temporais** (`Year_of_Transfer` e `Month_of_Transfer`) a partir da data de transação, de forma a capturar flutuações macroeconómicas e sazonais. Criámos também a **variável espacial** (`Postcode_Outcode`), isolando a primeira metade do código postal para capturar os efeitos fixos espaciais na modelação. Por fim, convertemos as variáveis qualitativas para o tipo categórico do Pandas.


In [ ]:
import pandas as pd

print("--- 5. Feature Engineering (Criar Variáveis do Word) ---")

# Vamos criar um novo dataframe chamado df_modelo a partir do teu df_clean
df_modelo = df_clean.copy()

# 1. Garantir que a Date_of_Transfer está no formato datetime
print("A processar datas...")
df_modelo['Date_of_Transfer'] = pd.to_datetime(df_modelo['Date_of_Transfer'])

# 2. Criar Variáveis Temporais (Year_of_Transfer e Month_of_Transfer)
df_modelo['Year_of_Transfer'] = df_modelo['Date_of_Transfer'].dt.year.astype('category')
df_modelo['Month_of_Transfer'] = df_modelo['Date_of_Transfer'].dt.month.astype('category')

# 3. Criar Variável Espacial (Postcode_Outcode)
# Como tiraste os espaços no Passo 4, o Outcode é tudo menos os últimos 3 caracteres!
print("A extrair a variável espacial (Postcode_Outcode)...")
df_modelo['Postcode_Outcode'] = df_modelo['Postcode_Clean'].astype(str).str[:-3].astype('category')

# 4. Ajustar outras variáveis categóricas
df_modelo['Property_Type'] = df_modelo['Property_Type'].astype('category')
df_modelo['CURRENT_ENERGY_RATING'] = df_modelo['CURRENT_ENERGY_RATING'].astype('category')
df_modelo['CONSTRUCTION_AGE_BAND'] = df_modelo['CONSTRUCTION_AGE_BAND'].astype('category')
df_modelo['Tenure_Duration'] = df_modelo['Tenure_Duration'].astype('category')

print("\n✅ Feature Engineering Concluída!")
print(f"O teu dataset 'df_modelo' está agora 100% alinhado com a secção 3.2.4 da tua tese.")

# Mostrar as novas colunas para confirmares
display(df_modelo[['Date_of_Transfer', 'Year_of_Transfer', 'Month_of_Transfer', 'Postcode_Clean', 'Postcode_Outcode']].head(5))

--- 5. Feature Engineering (Criar Variáveis do Word) ---
A processar datas...
A extrair a variável espacial (Postcode_Outcode)...

✅ Feature Engineering Concluída!
O teu dataset 'df_modelo' está agora 100% alinhado com a secção 3.2.4 da tua tese.


,Date_of_Transfer,Year_of_Transfer,Month_of_Transfer,Postcode_Clean,Postcode_Outcode
0,2021-01-01,2021,1,BR60EN,BR6
1,2021-01-01,2021,1,OL26JL,OL2
2,2021-01-01,2021,1,CR02XT,CR0
3,2021-01-01,2021,1,E28GW,E2
4,2021-01-01,2021,1,BR49PU,BR4


In [ ]:
df_modelo.head(3)

,Transaction_ID,Price,Date_of_Transfer,Postcode_Clean,Merge_Key,County,Property_Type,Old_New,Tenure_Duration,TOTAL_FLOOR_AREA,NUMBER_HABITABLE_ROOMS,CURRENT_ENERGY_RATING,CONSTRUCTION_AGE_BAND,Latitude,Longitude,Year_of_Transfer,Month_of_Transfer,Postcode_Outcode
0,{BEF7EBBF-3E58-7A76-E053-6B04A8C092F7},590000,2021-01-01,BR60EN,BR60EN_47,GREATER LONDON,S,N,F,104.18,5.0,E,England and Wales: 1900-1929,51.379582,0.099117,2021,1,BR6
1,{BC8936BC-4A47-0E2C-E053-6C04A8C0DBF4},116000,2021-01-01,OL26JL,OL26JL_10,GREATER MANCHESTER,O,N,F,73.00,4.0,D,England and Wales: 1983-1990,53.563072,-2.111319,2021,1,OL2
2,{E073986B-B017-2134-E053-6C04A8C0233B},290000,2021-01-01,CR02XT,CR02XT_145,GREATER LONDON,F,N,L,67.00,2.0,C,England and Wales: 2012 onwards,51.386351,-0.098812,2021,1,CR0


### 8. Guardar o dataset

In [ ]:
import os

print("--- 8. Guardar o Dataset Final (Backup) ---")

# Nome do ficheiro que vamos criar no teu Google Drive
nome_ficheiro_final = 'Dataset_Final_Tese.csv'
caminho_guardar = caminho_pasta + nome_ficheiro_final

print("A guardar o dataset em formato CSV... (Isto pode demorar 1 ou 2 minutos)")

# Guardar sem a coluna de índices do pandas
df_modelo.to_csv(caminho_guardar, index=False)

print(f"✅ SUCESSO! Dataset guardado no teu Drive em:\n{caminho_guardar}")
print(f"Tamanho final: {len(df_modelo)} linhas prontas para Machine Learning.")

--- 8. Guardar o Dataset Final (Backup) ---
A guardar o dataset em formato CSV... (Isto pode demorar 1 ou 2 minutos)
✅ SUCESSO! Dataset guardado no teu Drive em:
/content/drive/MyDrive/Tese/UK DataSets/Dataset_Final_Tese.csv
Tamanho final: 547948 linhas prontas para Machine Learning.
